# Multicollinearity and PCA — Severe Wasting (MICS Bangladesh)

This notebook complements `models_m.ipynb`. It examines predictor redundancy and PCA separately for (A) all children aged 0–4 and (B) children aged 0–1 completed years (under 2). It does **not** change or replace the prediction models. Run from the repository root and confirm the predictor lists against the final variable-selection cells in `models_m.ipynb` before interpreting results.

**Important:** PCA is unsupervised and does not use the outcome to construct components. One-hot encoding is used for categorical predictors; components are exploratory and are not automatically preferable to original variables.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

RANDOM_STATE = 42
pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')

## 1. Load the analysis data
The loader tries the common project-level dataset names. If your prepared analysis file has another name, set `DATA_PATH` explicitly. The notebook accepts CSV or Stata `.dta` files.

In [ ]:
CANDIDATE_PATHS = [
    Path('MICS-child-ISRC.dta'), Path('child_full.csv'),
    Path('data/child_full.csv'), Path('data/MICS-child-ISRC.dta')
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset not found. Set DATA_PATH to your prepared MICS file.')
print('Loading:', DATA_PATH)
if DATA_PATH.suffix.lower() == '.dta':
    df = pd.read_stata(DATA_PATH, convert_categoricals=False)
else:
    df = pd.read_csv(DATA_PATH, low_memory=False)
print('Shape:', df.shape)
display(df.head())

# Standardise column lookup while preserving the original dataframe.
column_lookup = {str(c).lower(): c for c in df.columns}
def find_col(name):
    return column_lookup.get(name.lower())

TARGET_COL = find_col('severe_wasting')
if TARGET_COL is None:
    whz_col = find_col('WHZ') or find_col('whz')
    if whz_col is not None:
        df['severe_wasting'] = (pd.to_numeric(df[whz_col], errors='coerce') < -3).astype('float')
        TARGET_COL = 'severe_wasting'
if TARGET_COL is None:
    raise KeyError('Could not find severe_wasting or WHZ. Confirm the outcome column.')
print('Outcome:', TARGET_COL, '\nOutcome counts (including missing):')
display(df[TARGET_COL].value_counts(dropna=False).to_frame('count'))

## 2. Define age-specific predictor sets
These lists use the project’s documented predictors and keep survey-design fields out of the feature matrix. Infant-only feeding and ANC variables are used only for the under-2 subset. **Edit the lists if `models_m.ipynb` uses a different final selected-variable set**; this notebook deliberately reports missing columns rather than silently substituting unrelated variables.

In [ ]:
# Core predictors documented in the project. Names are matched case-insensitively.
ALL_CHILDREN_CANDIDATES = [
    'age_years', 'age', 'ub2', 'sex', 'hl4', 'division', 'wealth_index', 'windex5',
    'welevel', 'mother_age', 'wb4', 'cm11', 'mstatus', 'maternal_education',
    'toilet_cat', 'fuel_cat', 'hh6', 'hh7', 'hh7a', 'hc8', 'eu4', 'ws1', 'ws11',
    'bd2', 'bd3', 'bd4', 'bd7a', 'bd7b', 'bd7d', 'bd7e', 'bd8a', 'bd8b',
    'bd8c', 'bd8d', 'bd8e', 'bd8f', 'bd8g', 'bd8h', 'bd8i', 'bd8j', 'bd8k',
    'bd8l', 'bd8m', 'bd8n', 'birth_order', 'number_of_births', 'diarrhoea', 'cough', 'breathing_difficulty'
]
INFANT_EXTRA_CANDIDATES = ['dds', 'fvs', 'fg_grain', 'fg_pulse', 'fg_dairy', 'fg_flesh', 'fg_egg', 'fg_vit_a', 'fg_other_fruits_vegetables', 'bf_status', 'bottle', 'anc_cat', 'bd9']

# Variables that must never be predictors: outcome, identifiers/design variables,
# and variables derived directly from WHZ / wasting outcome.
FORBIDDEN_NAMES = {
    'severe_wasting', 'whz', 'whz2', 'mod_wasting', 'any_wasting', 'wast_cat',
    'hh1', 'hh2', 'ln', 'chweight', 'strata', 'psu', 'cluster', 'id'
}

def resolve_columns(candidates):
    found, missing = [], []
    for name in candidates:
        actual = find_col(name)
        if actual is None: missing.append(name)
        elif str(actual).lower() not in FORBIDDEN_NAMES and actual not in found: found.append(actual)
    return found, missing

all_predictors, missing_all = resolve_columns(ALL_CHILDREN_CANDIDATES)
infant_extra, missing_infant = resolve_columns(INFANT_EXTRA_CANDIDATES)
print('All-children predictors found:', len(all_predictors))
print('Candidate names not found (all children):', missing_all)
print('Infant-specific predictors found:', len(infant_extra))
print('Candidate names not found (infant):', missing_infant)
display(pd.DataFrame({'all_children_predictors': pd.Series(all_predictors), 'infant_extra_predictors': pd.Series(infant_extra)}))

# Resolve age. Prefer the project's completed-age field if present.
AGE_COL = next((find_col(x) for x in ['age_years', 'ub2', 'age'] if find_col(x) is not None), None)
if AGE_COL is None:
    raise KeyError('Age column not found. Set AGE_COL to the completed-age-in-years variable.')
print('Age field used for under-2 filter:', AGE_COL)

# Remove generic age candidates from list if it is not the selected age column, then add chosen field once.
all_predictors = [c for c in all_predictors if c.lower() not in {'age_years','age','ub2'}]
all_predictors = [AGE_COL] + all_predictors
infant_predictors = list(dict.fromkeys(all_predictors + infant_extra))


## 3. Prepare comparable analysis samples
Missing values are imputed within preprocessing pipelines for diagnostic purposes only. The missingness table is retained because missingness may be structural, especially for infant-feeding fields. Do not interpret these diagnostic imputations as a final inferential or predictive preprocessing decision.

In [ ]:
def make_sample(data, predictors, infant=False):
    cols = list(dict.fromkeys(predictors))
    work = data.copy()
    if infant:
        age = pd.to_numeric(work[AGE_COL], errors='coerce')
        work = work.loc[age.isin([0, 1])].copy()
    work = work.loc[work[TARGET_COL].notna()].copy()
    usable = [c for c in cols if c in work.columns and c != TARGET_COL and c.lower() not in FORBIDDEN_NAMES]
    X = work[usable].copy()
    y = pd.to_numeric(work[TARGET_COL], errors='coerce')
    valid = y.notna() & y.isin([0, 1])
    X, y = X.loc[valid], y.loc[valid].astype(int)
    # Drop all-empty columns; they cannot be imputed or analysed.
    all_missing = X.columns[X.isna().all()].tolist()
    X = X.drop(columns=all_missing)
    return X, y, work.loc[valid], all_missing

X_all, y_all, sample_all, empty_all = make_sample(df, all_predictors, infant=False)
X_inf, y_inf, sample_inf, empty_inf = make_sample(df, infant_predictors, infant=True)
print(f'All children: n={len(X_all):,}; cases={int(y_all.sum()):,}; predictors={X_all.shape[1]}')
print(f'Under 2:      n={len(X_inf):,}; cases={int(y_inf.sum()):,}; predictors={X_inf.shape[1]}')
print('All-empty columns removed:', empty_all, empty_inf)

def missingness_table(X):
    out = pd.DataFrame({'missing_n': X.isna().sum(), 'missing_pct': 100*X.isna().mean(), 'unique_nonmissing': X.nunique(dropna=True)})
    return out.sort_values('missing_pct', ascending=False)
display(missingness_table(X_all))
display(missingness_table(X_inf))

## 4. Multicollinearity — mixed-type predictor overlap
For categorical predictors, inspect Cramér’s V; for numeric pairs, inspect absolute Pearson correlation. These are pairwise screens, not complete multivariable diagnostics. Perfectly duplicated or deterministically derived features should be separated (for example, `dds` and its constituent food-group indicators).

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.empty or min(tab.shape) < 2: return np.nan
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.to_numpy().sum()
    if n == 0: return np.nan
    phi2 = chi2 / n
    r, k = tab.shape
    # Bias-corrected Cramer's V
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1)) if n > 1 else 0
    rc = r - ((r-1)**2)/(n-1) if n > 1 else r
    kc = k - ((k-1)**2)/(n-1) if n > 1 else k
    denom = min(kc-1, rc-1)
    return np.sqrt(phi2corr/denom) if denom > 0 else np.nan

def variable_types(X):
    numeric = [c for c in X if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique(dropna=True) > 10]
    categorical = [c for c in X if c not in numeric]
    return numeric, categorical

def pairwise_overlap(X, label):
    numeric, categorical = variable_types(X)
    if len(numeric) >= 2:
        corr = X[numeric].apply(pd.to_numeric, errors='coerce').corr().abs()
        plt.figure(figsize=(max(8, len(numeric)*0.65), max(6, len(numeric)*0.55)))
        sns.heatmap(corr, cmap='vlag', vmin=0, vmax=1, annot=len(numeric)<=12, fmt='.2f')
        plt.title(f'{label}: absolute numeric correlation'); plt.tight_layout(); plt.show()
        pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
        display(pairs.head(20).rename('abs_correlation').to_frame())
    rows = []
    for i, a in enumerate(categorical):
        for b in categorical[i+1:]:
            rows.append((a,b,cramers_v(X[a].fillna('__MISSING__'), X[b].fillna('__MISSING__'))))
    if rows:
        display(pd.DataFrame(rows, columns=['variable_1','variable_2','cramers_v']).sort_values('cramers_v', ascending=False).head(30))

pairwise_overlap(X_all, 'All children')
pairwise_overlap(X_inf, 'Under 2')

## 5. Variance Inflation Factors (VIF)
VIF is calculated on the one-hot encoded design matrix after median/mode imputation, with one reference category dropped. This avoids the dummy-variable trap. VIFs for individual dummy columns should be interpreted cautiously; inspect groups of related variables as well. The common 5/10 thresholds are heuristics, not automatic deletion rules.

In [ ]:
def make_design_matrix(X):
    numeric, categorical = variable_types(X)
    transformers = []
    if numeric:
        transformers.append(('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric))
    if categorical:
        transformers.append(('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))]), categorical))
    prep = ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)
    arr = prep.fit_transform(X)
    names = prep.get_feature_names_out()
    mat = pd.DataFrame(arr, columns=names, index=X.index)
    # Remove constant columns and any remaining duplicate columns.
    mat = mat.loc[:, mat.nunique(dropna=False) > 1]
    mat = mat.loc[:, ~mat.T.duplicated()]
    return mat

def calculate_vif(X, label):
    design = make_design_matrix(X)
    if design.shape[1] < 2:
        print(f'{label}: not enough encoded columns for VIF'); return pd.DataFrame()
    # VIF can be unstable when the encoded matrix is rank-deficient; report infinities rather than hiding them.
    values = []
    arr = add_constant(design, has_constant='add')
    for i, col in enumerate(arr.columns):
        if col == 'const': continue
        try: vif = variance_inflation_factor(arr.to_numpy(dtype=float), i)
        except Exception: vif = np.inf
        values.append((col, vif))
    result = pd.DataFrame(values, columns=['encoded_feature','VIF']).sort_values('VIF', ascending=False)
    print(label, '| encoded columns:', design.shape[1])
    display(result.head(40))
    print('Features with VIF > 5:', int((result.VIF > 5).sum()), '| VIF > 10:', int((result.VIF > 10).sum()))
    return result

vif_all = calculate_vif(X_all, 'All children')
vif_inf = calculate_vif(X_inf, 'Under 2')

## 6. PCA on the predictor matrix
PCA requires numeric input, so categorical predictors are one-hot encoded and continuous predictors are standardized. Missing values are imputed within the pipeline. Components reflect predictor variance, **not** association with severe wasting. Survey weights are not used in this ordinary PCA, so results are descriptive rather than design-weighted population estimates.

In [ ]:
def run_pca(X, label, variance_target=0.80):
    design = make_design_matrix(X)
    if design.shape[1] < 2:
        print(f'{label}: not enough encoded columns for PCA'); return None
    pca = PCA(random_state=RANDOM_STATE)
    scores = pca.fit_transform(design)
    evr = pca.explained_variance_ratio_
    cumulative = np.cumsum(evr)
    n80 = int(np.searchsorted(cumulative, variance_target) + 1)
    summary = pd.DataFrame({'component': np.arange(1, len(evr)+1), 'explained_variance_ratio': evr, 'cumulative_variance': cumulative})
    display(summary.head(20))
    print(f'{label}: {n80} components explain at least {variance_target:.0%} of encoded predictor variance.')
    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(summary.component, summary.cumulative_variance, marker='o')
    ax.axhline(variance_target, linestyle='--', label=f'{variance_target:.0%} threshold')
    ax.set(xlabel='Number of components', ylabel='Cumulative explained variance', title=f'{label}: PCA cumulative variance')
    ax.legend(); plt.tight_layout(); plt.show()
    loadings = pd.DataFrame(pca.components_.T, index=design.columns, columns=[f'PC{i+1}' for i in range(len(evr))])
    for pc in range(min(5, len(evr))):
        top = loadings.iloc[:, pc].abs().sort_values(ascending=False).head(15).index
        print(f'Top absolute loadings for PC{pc+1}:')
        display(loadings.loc[top, [f'PC{pc+1}']].sort_values(f'PC{pc+1}', key=np.abs, ascending=False))
    return {'pca':pca, 'scores':scores, 'summary':summary, 'loadings':loadings, 'design_columns':design.columns.tolist(), 'n_components_target':n80}

pca_all = run_pca(X_all, 'All children (0–4)')
pca_inf = run_pca(X_inf, 'Under 2 (ages 0–1)')

## 7. Compare PCA structure across age groups
The two groups have different predictor sets by design, so component loadings are not directly comparable when infant-only predictors are present. Use this section to inspect the number of components needed and the strongest loadings within each analysis, not to claim identical components.

In [ ]:
comparison = []
for name, result in [('all_children_0_4', pca_all), ('under2_0_1', pca_inf)]:
    if result is not None:
        comparison.append({'analysis': name, 'encoded_features': len(result['design_columns']), 'components_for_80pct_variance': result['n_components_target'], 'variance_first_component': result['summary'].iloc[0]['explained_variance_ratio'], 'variance_first_5_components': result['summary'].head(5)['explained_variance_ratio'].sum()})
display(pd.DataFrame(comparison))

## 8. Interpretation checklist and cautions
- High VIF / high pairwise overlap suggests redundant predictors; it does not prove a variable is scientifically unnecessary.
- Consider removing or separating deterministic constructions (e.g., dietary diversity score `dds` versus its component food-group indicators). Do not include both in the same model unless there is a clear reason.
- PCA is primarily a dimension-reduction diagnostic. For epidemiologic interpretation, retaining original predictors is often preferable because components can be difficult to explain.
- PCA maximizes predictor variance, not outcome prediction or causal relevance. Any supervised selection must happen inside training folds to avoid leakage.
- This notebook does not calculate survey-design-adjusted VIF/PCA, inferential standard errors, or outcome-informed component selection.
- Before finalizing, reconcile the candidate lists above with the exact variable-selection process in `models_m.ipynb` and document any differences.

In [ ]:
# Optional: export diagnostic tables to a local outputs directory.
# Uncomment after reviewing the results.
# Path('outputs').mkdir(exist_ok=True)
# vif_all.to_csv('outputs/vif_all_children.csv', index=False)
# vif_inf.to_csv('outputs/vif_under2.csv', index=False)
# if pca_all is not None: pca_all['summary'].to_csv('outputs/pca_variance_all_children.csv', index=False)
# if pca_inf is not None: pca_inf['summary'].to_csv('outputs/pca_variance_under2.csv', index=False)